# TrustDrive MiniLab - 90-minute workshop

*Integrity-aware shared control for autonomous driving under perception uncertainty.*

This notebook walks through the five parts of the demo. The core loop needs only
`numpy` and `matplotlib`; the optional CNN perception path needs `torch`.

1. Autonomous lane keeping (vehicle + LQR)
2. Learned perception (image -> lane error)
3. Can we trust the AI? (ensemble uncertainty + NIS)
4. Integrity-aware shared control
5. Discussion

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from trustdrive import *
from trustdrive.road import RoadRenderer, RoadConditions
from trustdrive.perception import SyntheticPerception
from trustdrive.simulation import run_simulation, compute_metrics
np.set_printoptions(precision=3, suppress=True)

## Part I - Autonomous lane keeping  *(15 min)*

The plant is the kinematic lane-error model $x_{k+1}=f(x_k,u_k)$ with
$x=[e_y,e_\psi]$. A speed-scheduled **LQR** with curvature feed-forward keeps the lane.
We start off-centre ($e_y=0.3$ m) on a gently weaving road and watch it recover.

In [ ]:
frames = build_scenario('nominal', T=30.0)
log = run_simulation('proposed', frames, seed=0)
met = compute_metrics(log)
print('RMSE e_y = %.3f m,  max|e_y| = %.3f m,  lane exits = %d' % (
      met['RMSE_ey'], met['max_ey'], met['lane_exits']))
plt.figure(figsize=(8,3))
plt.plot(log['t'], log['e_y']); plt.axhline(0, color='k', lw=0.5)
plt.xlabel('time [s]'); plt.ylabel('lateral error $e_y$ [m]')
plt.title('Nominal lane keeping'); plt.grid(alpha=0.3); plt.show()

## Part II - Learned perception  *(20 min)*

The controller no longer sees the state directly - it must read it from a **camera
image**. We render small road images and (here) use the analytic
`SyntheticPerception` stand-in; the CNN ensemble (`trustdrive.perception.CNNPerception`)
is a drop-in replacement once trained with `experiments/train_perception.py`.

In [ ]:
renderer = RoadRenderer()
conds = [('clear', RoadConditions()),
         ('fog+blur', RoadConditions(brightness=0.6, blur=2.6, noise=0.03)),
         ('occlusion', RoadConditions(occlusion=0.45)),
         ('OOD curve', RoadConditions())]
kaps  = [0.012, 0.012, 0.012, 0.05]
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
rng = np.random.default_rng(0)
for a,(name,c),k in zip(ax, conds, kaps):
    img = renderer.render(0.3, 0.02, k, c, rng)
    a.imshow(img, cmap='gray', vmin=0, vmax=1); a.set_title(name); a.axis('off')
plt.show()

In [ ]:
# image -> estimate (synthetic perception with reported uncertainty)
per = SyntheticPerception(seed=1)
clear = per.measure([0.3, 0.02], 0.012, RoadConditions())
foggy = per.measure([0.3, 0.02], 0.012, RoadConditions(brightness=0.6, blur=2.6))
for name, po in [('clear', clear), ('fog+blur', foggy)]:
    std = np.sqrt(np.diag(po.cov))
    print('%-9s estimate=[% .3f % .3f]  reported std=[%.3f m, %.3f rad]' %
          (name, po.mean[0], po.mean[1], std[0], std[1]))

## Part III - Can we trust the AI?  *(20 min)*

Two trust signals: the **ensemble covariance** $\Sigma^{AI}$ (epistemic uncertainty)
and the model-based **normalised innovation squared**
$\mathrm{NIS}=\nu^\top S^{-1}\nu$ compared to a $\chi^2_2$ threshold.

We deliberately blur the camera (`fog`) and push past the trained curvature
(`ood_curve`) and watch the three curves $e_y$, $\sigma_{AI}$, NIS.

In [ ]:
from trustdrive.kalman import CHI2_2DOF
for sc in ['fog', 'ood_curve']:
    log = run_simulation('proposed', build_scenario(sc, T=30.0), seed=0)
    t = log['t']
    fig, ax = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
    ax[0].plot(t, log['e_y']); ax[0].set_ylabel('$e_y$ [m]')
    ax[1].plot(t, np.sqrt(log['cov_trace']), color='purple')
    ax[1].set_ylabel(r'$\sqrt{tr\,\Sigma^{AI}}$')
    ax[2].plot(t, log['nis'], color='orange')
    ax[2].axhline(CHI2_2DOF[0.99], ls='--', color='k')
    ax[2].set_ylabel('NIS'); ax[2].set_xlabel('time [s]')
    for a in ax: a.grid(alpha=0.3)
    fig.suptitle(sc); plt.tight_layout(); plt.show()

**Observe:** in `fog` the uncertainty $\sigma_{AI}$ rises. In `ood_curve` the image
still looks valid so $\sigma_{AI}$ hardly moves - but the **NIS spikes above the
threshold**. Uncertainty alone would miss the out-of-distribution regime.

## Part IV - Integrity-aware shared control  *(20 min)*

The integrity score $I=f(\Sigma^{AI},\mathrm{NIS},\mathrm{ODD})$ drives the authority
$\delta=\delta_{ff}+\lambda\,fb_A+(1-\lambda)\,\delta_H$ and contracts the speed.
We compare **naive**, **hard handover** and the **proposed** system.

In [ ]:
rows = []
for sc in SCENARIOS:
    frames = build_scenario(sc, T=30.0)
    for m in METHODS:
        met = compute_metrics(run_simulation(m, frames, seed=0))
        rows.append((sc, m, met['RMSE_ey'], met['lane_exits'],
                     met['steering_jerk'], met['unsafe_ai_usage_pct']))
print('%-10s %-9s %8s %6s %8s %8s' % ('scenario','method','RMSE','exits','jerk','unsafe%'))
for r in rows:
    print('%-10s %-9s %8.3f %6d %8.2f %8.1f' % r)

In [ ]:
# authority + steering for the proposed system under fog
log = run_simulation('proposed', build_scenario('fog', T=30.0), seed=0)
t = log['t']
fig, ax = plt.subplots(2,1, figsize=(8,5), sharex=True)
ax[0].plot(t, log['lambda'], color='green'); ax[0].set_ylabel('authority $\\lambda$')
ax[0].plot(t, log['integrity'], color='navy', ls='--', label='integrity $I$'); ax[0].legend()
ax[1].plot(t, log['delta_auto'], label='$\\delta_A$')
ax[1].plot(t, log['delta_human'], label='$\\delta_H$', alpha=0.7)
ax[1].plot(t, log['delta'], 'k', label='applied'); ax[1].legend()
ax[1].set_ylabel('steering [rad]'); ax[1].set_xlabel('time [s]')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part V - Discussion  *(15 min)*

- Is uncertainty enough, or do we need model-based consistency (NIS)?
- How should an ODD be defined and monitored online?
- Should authority depend only on AI reliability, or also on driver state?
- How do we estimate workload and attention?
- Can integrity *guarantees* be provided rather than heuristics?

These questions lead directly into the trustworthy interactive autonomous driving
research agenda: model-based estimation, integrity monitoring, LPV control and
shared human-vehicle control.